In [3]:
import openpyxl as wb
import pandas as pd
from geopy.geocoders import Nominatim
import folium
from geopy.extra.rate_limiter import RateLimiter

In [5]:
planilhaLinks = wb.load_workbook("SPTC_12Meses_2026-01-30.XLSX")
#print(planilhaLinks.sheetnames)
aba = planilhaLinks["PA_em_02-02-2026"]
colunas = [3,5,6,29]
dados = []
for row in aba.iter_rows(values_only=True):
    linhas = [row[i-1] for i in colunas if i-1 <len(row)]
    if any(linhas):
        dados.append(linhas)
df = pd.DataFrame(dados[1:], columns=dados[0])
df.columns = ["Unidade", "CEP","Endereço", "IP"]
print(df.head(10))
   
    

                                             Unidade        CEP  \
0        EQUIPE DE PERICIA CRIMINALISTICA-ADAMANTINA  17800-005   
1         EQUIPE DE PERICIA CRIMINALISTICA-AMERICANA  13478-800   
2         EQUIPE DE PERICIA CRIMINALISTICA-ANDRADINA  16900-411   
3             EQUIPE DE PERICIA CRIMINALISTICA-ASSIS  19813-173   
4             EQUIPE DE PERICIA CRIMINALISTICA-AVARE  18705-390   
5          EQUIPE DE PERICIA CRIMINALISTICA-BARRETOS  14780-755   
6          EQUIPE DE PERICIA CRIMINALISTICA-BARRETOS  14780-100   
7         EQUIPE DE PERICIA CRIMINALISTICA-BEBEDOURO  14700-420   
8          EQUIPE DE PERICIA CRIMINALISTICA-BOTUCATU  18608-550   
9  EQUIPE DE PERICIA CRIMINALISTICA-BRAGANCA PAUL...  12916-000   

                 Endereço            IP  
0     ALAMEDA FERNAO DIAS   10.78.184.1  
1  AVENIDA ANGELO PASCOTE   10.78.254.1  
2       RUA SAO FRANCISCO   10.78.201.1  
3  RUA BIAGGIO DE FILIPPO   10.78.173.1  
4         RUA MATO GROSSO   10.78.246.1  
5         

In [6]:
geolocator = Nominatim(user_agent="meu_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
df['endereco_completo'] = df["Endereço"] + ', ' + df['CEP'] + ', São Paulo, Brasil'
df['local'] = df['endereco_completo'].apply(geocode)
df['latitude'] = df['local'].apply(lambda x: x.latitude if x else None)
df['longitude'] = df['local'].apply(lambda x: x.longitude if x else None)


In [ ]:
m = folium.Map(location=[-23.5505, -46.6333], zoom_start=11)
for idx, row in df.dropna(subset=['latitude']).iterrows():
    folium.Marker(
        [row['latitude'], row['longitude']],
        popup=f"{row['Endereço']}<br>CEP: {row['CEP']}",
        tooltip=row['Unidade'] + row['IP']
    ).add_to(m)

m.save('mapa_enderecos.html') 
m


TypeError: can only concatenate str (not "list") to str